In [ ]:
!pip install git+https://github.com/huggingface/transformers.git

In [14]:
from PIL import Image
import os
import pandas as pd
from tqdm.auto import tqdm
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

tqdm.pandas()

IMAGE_FOLDER = "/kaggle/input/avito-purple-hack/dataset_colors/test_data"   
CSV_PATH = "/kaggle/input/avito-purple-hack/dataset_colors/test_data.csv"
OUTPUT_PATH = "submission.csv"

In [ ]:
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct", torch_dtype="auto", device_map="auto"
)

processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")

In [26]:
def ask_qwen2_5_vl(image_path, question):
    image = Image.open(image_path).convert('RGB')

    messages = [
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": question}
        ]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = processor(images=image, text=text, return_tensors="pt").to(model.device)

    generated_ids = model.generate(**inputs)

    answer = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return answer


In [27]:
def process_images_with_qwen(image_folder: str, csv_path: str, output_path: str):
    df = pd.read_csv(csv_path).sort_values(by='id')
    
    if "id" not in df.columns or "category" not in df.columns:
        raise ValueError("CSV-файл должен содержать колонки 'id' и 'category'")
    
    data_list = []
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing images"):
        image_id = row["id"]
        category = row["category"]
        formatted_question = f"""
        Identify the color of the object in the photo, which belongs to the category: {category}.
        Choose only one color from the following list: 
        ['green', 'black', 'white', 'maroon', 'red', 'beige', 'multicolored', 'pink', 'silver', 'brown', 
         'purple', 'gray', 'light blue', 'orange', 'blue', 'turquoise', 'gold', 'yellow'].
        Output only the color name.
        """
        image_path_png = os.path.join(image_folder, f"{image_id}.png")
        image_path_jpg = os.path.join(image_folder, f"{image_id}.jpg")

        if os.path.exists(image_path_png):
            image_path = image_path_png
        elif os.path.exists(image_path_jpg):
            image_path = image_path_jpg
        else:
            print(f"Изображение {image_id} не найдено.")
            continue

        response = ask_qwen2_5_vl(image_path, formatted_question).split('assistant')[-1][1:]
        data_list.append({"ID": image_id, "category": category, "response": response})

    result_df = pd.DataFrame(data_list)
    result_df.to_csv(output_path, index=False, encoding="utf-8")
    print(f"Результаты сохранены в {output_path}")
    return result_df


In [ ]:
result_df = process_images_with_qwen(IMAGE_FOLDER, CSV_PATH, OUTPUT_PATH)


In [15]:
color_dict = {
    'green': 'zelenyi',
    'black': 'chernyi',
    'white': 'belyi',
    'maroon': 'bordovyi',
    'red': 'krasnyi',
    'beige': 'bezhevyi',
    'multicolored': 'raznocvetnyi',
    'pink': 'rozovyi',
    'silver': 'serebristyi',
    'brown': 'korichnevyi',
    'purple': 'fioletovyi',
    'gray': 'seryi',
    'light blue': 'goluboi',
    'orange': 'oranzhevyi',
    'blue': 'sinii',
    'turquoise': 'biryuzovyi',
    'gold': 'zolotoi',
    'yellow': 'zheltyi',
    "I don't know.": "ood" 
}
result_df['predict_color'] = result_df['response'].apply(lambda x: x.lower()).map(color_dict)
result_df['predict_color'] = result_df['predict_color'].fillna('raznocvetnyi')
result_df['predict_proba'] = result_df['response'].apply(lambda x: {color_dict[color]: 1.0 if color == x.lower() else 0.0 for color in color_dict.keys()})
result_df.to_csv(OUTPUT_PATH, index=False)

In [16]:
result_df

,ID,category,response,predict_color,predict_proba
0,19564221134,clothes,white,belyi,"{'zelenyi': 0.0, 'chernyi': 0.0, 'belyi': 1.0,..."
1,19757410590,clothes,light blue,goluboi,"{'zelenyi': 0.0, 'chernyi': 0.0, 'belyi': 0.0,..."
2,19758386597,clothes,red,krasnyi,"{'zelenyi': 0.0, 'chernyi': 0.0, 'belyi': 0.0,..."
3,19759567006,clothes,pink,rozovyi,"{'zelenyi': 0.0, 'chernyi': 0.0, 'belyi': 0.0,..."
4,19762915377,clothes,green,zelenyi,"{'zelenyi': 1.0, 'chernyi': 0.0, 'belyi': 0.0,..."
...,...,...,...,...,...
1429,37627097645,chairs,turquoise,biryuzovyi,"{'zelenyi': 0.0, 'chernyi': 0.0, 'belyi': 0.0,..."
1430,37627213611,chairs,pink,rozovyi,"{'zelenyi': 0.0, 'chernyi': 0.0, 'belyi': 0.0,..."
1431,37627213936,chairs,maroon,bordovyi,"{'zelenyi': 0.0, 'chernyi': 0.0, 'belyi': 0.0,..."
1432,37627244884,table,beige,bezhevyi,"{'zelenyi': 0.0, 'chernyi': 0.0, 'belyi': 0.0,..."
